# **Exercise 4: Neural Language Modelling**

In this exercise, you will implement the core components of a small decoder-only Transformer language model.

We will follow the slides from week 4. We will see:

1. scaled dot-product attention;
2. causal masking;
3. multi-head attention;
4. a decoder block;
5. shifted next-token prediction; and
6. autoregressive generation.


### Google Colab setup

This notebook trains a Transformer. In Google Colab, enable a GPU before running it:

1. Open **Runtime -> Change runtime type**.
2. Select **T4 GPU** under *Hardware accelerator*.
3. Click **Save**, then run the notebook from the beginning.

The notebook can also run on CPU (with fewer training steps), but the generated text will be less coherent.

Relevant *Speech and Language Processing* material:

- [Chapter 7: Neural Networks and Neural Language Models](https://web.stanford.edu/~jurafsky/slp3/7.pdf)
- [Chapter 8: Transformers](https://web.stanford.edu/~jurafsky/slp3/8.pdf)


In [ ]:
%pip install -q datasets tokenizers tqdm

import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

np.random.seed(1)
torch.manual_seed(1)
torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cpu":
    print("For faster training in Colab, select Runtime -> Change runtime type -> T4 GPU")

## **Ex. 1: Scaled dot-product attention**

Self-attention creates a contextual representation of each token by combining information from other positions.

For an input matrix \(X\), we make projections that produce queries, keys, and values:

$$
Q = XW_Q, \qquad K = XW_K, \qquad V = XW_V
$$

Scaled dot-product attention is:

$$
\operatorname{Attention}(Q,K,V)
= \operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

In this first exercise, the token encodings and projection matrices are fixed so that we can inspect the resulting attention weights.


In [ ]:
sentence_1 = "Denver lost from Minnesota because they were too strong".split()
sentence_2 = "Denver lost from Minnesota because they were too weak".split()

# Shape: sequence_length x model_dimension
encoding_1 = np.array(
    [
        [1, 0],
        [0, 0],
        [0, 0],
        [-1, 0],
        [0, 0],
        [0, 1],
        [0, 0],
        [0, 0],
        [-1, 1],
    ],
    dtype=float,
)
encoding_2 = np.array(
    [
        [1, 0],
        [0, 0],
        [0, 0],
        [-1, 0],
        [0, 0],
        [0, 1],
        [0, 0],
        [0, 0],
        [1, 1],
    ],
    dtype=float,
)

W_q = np.array([[1, 0, 0], [0, 1, 0]], dtype=float)
W_k = np.array([[1, 0, 0], [0, 1, 0]], dtype=float)
W_v = np.eye(2)


def softmax(values, axis=-1):
    values = values - np.max(values, axis=axis, keepdims=True)
    exponentials = np.exp(values)
    return exponentials / exponentials.sum(axis=axis, keepdims=True)

Implement one self-attention pass. Keep track of the matrix shapes and apply softmax over the key dimension.

<div class="alert alert-block alert-info">
<b>TIME:</b> About 15 minutes, including interpretation of the plots.
</div>


In [ ]:
def attend(encoding):
    # Project the input into queries, keys, and values.
    queries = ...
    keys = ...
    values = ...

    # Compute scaled scores, attention weights, and contextual outputs.
    scores = ...
    attention_weights = ...
    attention_output = ...

    return attention_weights, attention_output

In [ ]:
def plot_attention(attention_weights, words, query_index, title):
    plt.figure(figsize=(9, 2))
    plt.imshow(attention_weights[query_index][None, :], cmap="Greens", aspect="auto")
    plt.xticks(range(len(words)), words, rotation=30)
    plt.yticks([0], [words[query_index]])
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.show()


they_index = 5
for number, (sentence, encoding) in enumerate(
    [(sentence_1, encoding_1), (sentence_2, encoding_2)],
    start=1,
):
    first_weights, first_output = attend(encoding)
    second_weights, second_output = attend(first_output)

    plot_attention(
        first_weights,
        sentence,
        they_index,
        f"Sentence {number}: first attention layer",
    )
    plot_attention(
        second_weights,
        sentence,
        they_index,
        f"Sentence {number}: second attention layer",
    )
    print(f"Sentence {number}, contextualized 'they':", second_output[they_index])

**Discuss briefly:** What changes between the first and second attention layers? Compare the attention paid to `Denver` and `Minnesota` in the two sentences. Why should attention weights not automatically be treated as explanations of a model's decision?


## **Ex. 2: Causal masking**

The previous attention operation is bidirectional: every position can attend to every other position. A language model must not use future tokens when predicting the next token.

For a sequence of four tokens, a causal mask can allow the following connections:

```text
position 0: 0
position 1: 0, 1
position 2: 0, 1, 2
position 3: 0, 1, 2, 3
```

Implement a Boolean lower-triangular mask. `True` means that attention is allowed.

<div class="alert alert-block alert-info">
<b>TIME:</b> About 10 minutes.
</div>


In [ ]:
def causal_mask(sequence_length: int, device=None) -> torch.Tensor: ...

In [ ]:
mask = causal_mask(6)
print(mask.int())

example_scores = torch.arange(36, dtype=torch.float).view(6, 6)
masked_scores = example_scores.masked_fill(~mask, float("-inf"))
print(torch.softmax(masked_scores, dim=-1))

**Discuss briefly:** What information leaks without this mask? How is a causal mask different from a padding mask? Why does a standard Transformer encoder normally use bidirectional attention?


## **Ex. 3: Multi-head attention**

A multi-head attention layer applies several attention operations in parallel. Each head works in a smaller representation space of size:

$$
d_k = \frac{d_{model}}{h}
$$

The heads are concatenated and projected back to `d_model`.

Complete `scaled_dot_product_attention()` and `forward()`. The helper methods for splitting and combining heads are provided.

<div class="alert alert-block alert-info">
<b>TIME:</b> About 20 minutes.
</div>


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.query_projection = nn.Linear(d_model, d_model)
        self.key_projection = nn.Linear(d_model, d_model)
        self.value_projection = nn.Linear(d_model, d_model)
        self.output_projection = nn.Linear(d_model, d_model)

    def split_heads(self, tensor):
        batch_size, sequence_length, _ = tensor.shape
        tensor = tensor.view(batch_size, sequence_length, self.num_heads, self.d_k)
        return tensor.transpose(1, 2)

    def combine_heads(self, tensor):
        batch_size, _, sequence_length, _ = tensor.shape
        tensor = tensor.transpose(1, 2).contiguous()
        return tensor.view(batch_size, sequence_length, self.d_model)

    def scaled_dot_product_attention(self, queries, keys, values, mask=None): ...

    def forward(self, queries, keys, values, mask=None): ...

In [ ]:
batch_size = 2
sequence_length = 5
d_model = 16

attention = MultiHeadAttention(d_model=d_model, num_heads=4)
example_input = torch.randn(batch_size, sequence_length, d_model)
mask = causal_mask(sequence_length)
example_output = attention(example_input, example_input, example_input, mask)

print("MultiHeadAttention output shape:", example_output.shape)

## **Ex. 4: A decoder block**

A decoder-only language model stacks blocks containing:

1. masked self-attention;
2. a residual connection;
3. layer normalization;
4. a position-wise feedforward network; and
5. another residual connection and normalization.

The implementation below uses the common **pre-norm** ordering: normalize the input before each sublayer.

Complete the forward pass.

<div class="alert alert-block alert-info">
<b>TIME:</b> About 15 minutes.
</div>


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, inputs):
        return self.layers(inputs)


class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.feedforward = FeedForward(d_model, d_ff, dropout)
        self.norm_1 = nn.LayerNorm(d_model)
        self.norm_2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs, mask): ...

A complete language model also includes token embeddings, learned position embeddings, several decoder blocks, a final normalization layer, and a vocabulary projection.

Most of this code is provided below, but do make sure your decoder block is working correctly!


In [ ]:
class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_sequence_length, dropout):
        super().__init__()
        self.max_sequence_length = max_sequence_length
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_sequence_length, d_model)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.output_projection = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, token_ids):
        _, sequence_length = token_ids.shape
        if sequence_length > self.max_sequence_length:
            raise ValueError("Input sequence is longer than max_sequence_length")

        positions = torch.arange(sequence_length, device=token_ids.device)
        hidden = self.token_embedding(token_ids) + self.position_embedding(positions)
        hidden = self.dropout(hidden)

        mask = causal_mask(sequence_length, device=token_ids.device)
        for block in self.blocks:
            hidden = block(hidden, mask)

        return self.output_projection(self.final_norm(hidden))

In [ ]:
# Before training, verify that all model components work together in both
# the forward and backward passes.
test_model = TinyTransformerLM(
    vocab_size=50,
    d_model=16,
    num_heads=4,
    num_layers=2,
    d_ff=32,
    max_sequence_length=8,
    dropout=0.0,
)
test_token_ids = torch.randint(0, 50, (2, 8))
test_targets = torch.randint(0, 50, (2, 8))
test_logits = test_model(test_token_ids)
test_loss = nn.CrossEntropyLoss()(
    test_logits.reshape(-1, 50),
    test_targets.reshape(-1),
)
test_loss.backward()

# If this does not pass, you have an error somewhere above!
assert test_logits.shape == (2, 8, 50)
assert torch.isfinite(test_logits).all()
assert test_model.output_projection.weight.grad is not None
print("Pre-training model check passed.")

## **Ex. 5: Tokenization and next-token training**

We will train a small BPE tokenizer with the Hugging Face `tokenizers` library (like we've seen in Exercise 1).

The tokenizer reserves four special tokens:

- `[PAD]` for padding;
- `[UNK]` for unknown input;
- `[BOS]` for the beginning of a sequence; and
- `[EOS]` for the end of a sequence.


In [ ]:
text_corpus = " ".join(load_dataset("winglian/tiny-shakespeare")["train"]["text"])
print(f"Corpus size: {len(text_corpus):,} characters")
print(repr(text_corpus[:300]))

In [ ]:
special_tokens = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]

hf_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
hf_tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()
hf_tokenizer.decoder = decoders.Metaspace()
trainer = trainers.BpeTrainer(vocab_size=500, special_tokens=special_tokens)
hf_tokenizer.train_from_iterator([text_corpus], trainer=trainer)

pad_id = hf_tokenizer.token_to_id("[PAD]")
bos_id = hf_tokenizer.token_to_id("[BOS]")
eos_id = hf_tokenizer.token_to_id("[EOS]")
vocab_size = hf_tokenizer.get_vocab_size()

print("Vocabulary size:", vocab_size)
print(hf_tokenizer.encode("To be, or not to be.").tokens)

For next-token prediction, a sequence is shifted by one position:

```text
Input:  [BOS] the cat sat on the mat
Target: the   cat sat on  the mat [EOS]
```

The dataset below consists of fixed-length windows from the tokenized corpus. Complete the training function by creating the shifted inputs and targets.

<div class="alert alert-block alert-info">
<b>TIME:</b> About 30 minutes, including training and discussion.
</div>


In [ ]:
sequence_length = 64
corpus_ids = [bos_id, *hf_tokenizer.encode(text_corpus).ids, eos_id]

# We take a sample from the dataset to keep the training time manageable
corpus_ids = corpus_ids[:500_000]
windows = [
    corpus_ids[start : start + sequence_length + 1]
    for start in range(0, len(corpus_ids) - sequence_length, sequence_length)
]
training_tensor = torch.tensor(windows, dtype=torch.long)
training_loader = DataLoader(
    TensorDataset(training_tensor),
    batch_size=32,
    shuffle=True,
    pin_memory=device.type == "cuda",
)

print("Training windows:", len(training_tensor))
print("Window shape:", training_tensor.shape)

In [ ]:
def train_language_model(model, data_loader, optimizer, criterion, device, max_steps):
    model.train()
    losses = []
    step = 0
    progress = tqdm(total=max_steps)

    # Continue over the dataset for as many epochs as necessary.
    while step < max_steps:
        for (batch,) in data_loader:
            batch = batch.to(device)

            # Shift the sequence for next-token prediction.
            model_input = ...
            expected_output = ...

            optimizer.zero_grad()
            logits = ...
            loss = ...
            ...

            losses.append(loss.item())
            step += 1
            progress.update(1)
            progress.set_postfix(loss=f"{loss.item():.3f}")
            if step >= max_steps:
                break

    progress.close()
    return losses

In [ ]:
# We change the parameters depending on whether a GPU is available.
# This keeps training time manageable.
using_gpu = device.type == "cuda"

model_dimension = 128 if using_gpu else 64
number_of_heads = 4
number_of_layers = 4 if using_gpu else 2
feedforward_dimension = 512 if using_gpu else 256
model_dropout = 0.1
learning_rate = 3e-4
weight_decay = 0.01
max_training_steps = 2_000 if using_gpu else 150

language_model = TinyTransformerLM(
    vocab_size=vocab_size,
    d_model=model_dimension,
    num_heads=number_of_heads,
    num_layers=number_of_layers,
    d_ff=feedforward_dimension,
    max_sequence_length=sequence_length,
    dropout=model_dropout,
).to(device)

optimizer = torch.optim.AdamW(
    language_model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

print(
    f"Model configuration: d_model={model_dimension}, "
    f"heads={number_of_heads}, layers={number_of_layers}, "
    f"d_ff={feedforward_dimension}"
)
print(f"Optimization: learning_rate={learning_rate}, weight_decay={weight_decay}, steps={max_training_steps}")

losses = train_language_model(
    language_model,
    training_loader,
    optimizer,
    criterion,
    device,
    max_steps=max_training_steps,
)

print(f"Training steps: {len(losses)}")
print(f"Initial loss:   {losses[0]:.3f}")
print(f"Final loss:     {losses[-1]:.3f}")
print(f"Final batch perplexity: {math.exp(losses[-1]):.2f}")

plt.plot(losses)
plt.xlabel("Training step")
plt.ylabel("Cross-entropy loss")
plt.show()

## **Ex. 6: Autoregressive generation**

At inference time, the model does not receive the "true" continuation. It repeatedly predicts one new token and appends that token to its own input.

Complete top-k sampling below. The "temperature" controls how peaked/flat the distribution is. The `top_k` sampling only selects the most likely candidates. Keep only the most recent `max_sequence_length` tokens as model input.

<div class="alert alert-block alert-info">
<b>TIME:</b> About 10 minutes, including the final discussion.
</div>


In [ ]:
def generate(
    model,
    tokenizer,
    prompt,
    bos_id,
    eos_id,
    device,
    max_new_tokens=60,
    temperature=0.7,
    top_k=20,
):
    model.eval()
    generated_ids = [bos_id, *tokenizer.encode(prompt).ids]

    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = ...
            logits = ...

            # Apply temperature and retain only the top-k candidates.
            next_token_logits = ...
            top_values, top_indices = ...
            probabilities = ...
            sampled_index = ...
            next_token = ...
            ...

    return tokenizer.decode(generated_ids, skip_special_tokens=True)

In [ ]:
for prompt in ("ROMEO:", "KING:", "To be, or not to be"):
    print(
        generate(
            language_model,
            hf_tokenizer,
            prompt=prompt,
            bos_id=bos_id,
            eos_id=eos_id,
            device=device,
            max_new_tokens=60,
            temperature=0.7,
            top_k=20,
        )
    )
    print("---")

**Discuss briefly:**

1. Why can the training loss decrease while generated text remains poor?
2. What is the difference between teacher forcing and autoregressive generation?
3. Why is perplexity meaningful only relative to a particular tokenizer and dataset?
4. How does the sampling and decoding from the third session (i.e. beam search) relate to the ones used here?


## **Advanced: Encoder-only, decoder-only, and encoder–decoder models**

How do the masking patterns and training objectives differ between these architectures?

<div class="alert alert-block alert-info">
<b>TIME:</b> About 10 minutes. This section is optional.
</div>


## **Takeaways**

- Attention combines values according to query-key similarities.
- Scaling and softmax influence how the attention distribution looks like.
- Causal masking prevents future tokens from being seen.
- Multi-head attention allows several projections to be learned in parallel.
- Transformer blocks combine the attention, feedforward layers, residual connections, and normalization.
- Next-token training requires shifting the input and target sequences.
- Autoregressive generation uses / conditions on the model's own previous predictions.
